In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as pyg
import torch_geometric.nn as nn_g
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
import numpy as np
import networkx as nx

from wsi.wsi import WSI, WSI_Graph
from wsi.train import Trainer
from wsi.utils import *
from wsi.nets.vision_transformer import vit_small
from wsi.nets.blocks import Classify_Head
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2 as cv

In [ ]:
normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])

In [ ]:
# # 准备数据 Camelyon16

# img_level = 5
# level = 3

# level_base = 5
# p = 2**(level_base-level)
# p_size = 256
# w_size = p_size//p
# batch_size=64

# device = torch.device("cuda:0")
# patch_size=16
# model = vit_small(patch_size=patch_size)
# state_dict = torch.load(f"./model/base/DINO/{img_level}_checkpoint.pth")["teacher"]
# state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
# state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
# model.load_state_dict(state_dict,strict=False)
# model.eval()
# model = model.to(device)

# # train
# wsi_path = "../../CAMELYON16/training/normal/"
# save_path = f"../../CAMELYON16/pkl_data/training/{img_level}xbags/"
# for f in tqdm(os.listdir(wsi_path)):

#     wsi = WSI(os.path.join(wsi_path,f))
#     _,rects,_ = wsi.get_tissue_patches(level,level_base,p_size,0.5)

#     feature_list = []
#     for i in range(0,len(rects),batch_size):
#         with torch.no_grad():
#             batch = rects[i:i+batch_size]
#             patches  = torch.stack([normalize(wsi.read_region(r,level,(p_size,p_size))) for r in batch])
#             y = model(patches.to(device))
#             feature_list.append(y)

#     feature_list = torch.concat(feature_list,0)
#     label = 0

#     torch.save({
#         "features":feature_list,
#         "rects":rects,
#         "level":level,
#         "level_base":level_base,
#         "patch_size":p_size,
#         "level":level,
#         "label":label
#     },os.path.join(save_path,f"{f[:-4]}_{len(feature_list)}_{label}.pth"))

# wsi_path = "../../CAMELYON16/training/tumor/"
# save_path = f"../../CAMELYON16/pkl_data/training/{img_level}xbags/"
# for f in tqdm(os.listdir(wsi_path)):

#     wsi = WSI(os.path.join(wsi_path,f))
#     _,rects,_ = wsi.get_tissue_patches(level,level_base,p_size,0.5)

#     feature_list = []
#     for i in range(0,len(rects),batch_size):
#         with torch.no_grad():
#             batch = rects[i:i+batch_size]
#             patches  = torch.stack([normalize(wsi.read_region(r,level,(p_size,p_size))) for r in batch])
#             y = model(patches.to(device))
#             feature_list.append(y)

#     feature_list = torch.concat(feature_list,0)
#     label = 1

#     torch.save({
#         "features":feature_list,
#         "rects":rects,
#         "level":level,
#         "level_base":level_base,
#         "patch_size":p_size,
#         "level":level,
#         "label":label
#     },os.path.join(save_path,f"{f[:-4]}_{len(feature_list)}_{label}.pth"))



# # test
# wsi_path = "../../CAMELYON16/testing/images/"
# save_path = f"../../CAMELYON16/pkl_data/testing/{img_level}xbags/"
# tumor_set = set([f[:-4].split("_")[1] for f in os.listdir("../../CAMELYON16/label/testing")])
# for f in tqdm(os.listdir(wsi_path)):

#     wsi = WSI(os.path.join(wsi_path,f))
#     _,rects,_ = wsi.get_tissue_patches(level,level_base,p_size,0.5)

#     feature_list = []
#     for i in range(0,len(rects),batch_size):
#         with torch.no_grad():
#             batch = rects[i:i+batch_size]
#             patches  = torch.stack([normalize(wsi.read_region(r,level,(p_size,p_size))) for r in batch])
#             y = model(patches.to(device))
#             feature_list.append(y)

#     feature_list = torch.concat(feature_list,0)
#     label = 1 if f[:-4].split("_")[1] in tumor_set else 0

#     torch.save({
#         "features":feature_list,
#         "rects":rects,
#         "level":level,
#         "level_base":level_base,
#         "patch_size":p_size,
#         "level":level,
#         "label":label
#     },os.path.join(save_path,f"{f[:-4]}_{len(feature_list)}_{label}.pth"))





In [ ]:
# # 准备数据 BACH

# img_level = 20
# level = 0

# level_base = 2
# p = 4**(level_base-level)
# p_size = 256
# w_size = p_size//p
# batch_size=64

# device = torch.device("cuda:0")
# patch_size=16
# model = vit_small(patch_size=patch_size)
# state_dict = torch.load(f"./model/base/DINO/{img_level}_checkpoint.pth")["teacher"]
# state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
# state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
# model.load_state_dict(state_dict,strict=False)
# model.eval()
# model = model.to(device)

# # train
# wsi_path = "../../BACH/WSI/train/wsi/"
# save_path = f"../../BACH/pkl_data/training/{img_level}xbags/"
# for f in tqdm([f for f in os.listdir(wsi_path) if f.endswith("svs")][-9:]):
#     wsi = WSI(os.path.join(wsi_path,f))
#     _,rects,_ = wsi.get_tissue_patches(level,level_base,p_size,0.5)

#     feature_list = []
#     for i in range(0,len(rects),batch_size):
#         with torch.no_grad():
#             batch = rects[i:i+batch_size]
#             patches  = torch.stack([normalize(wsi.read_region(r,level,(p_size,p_size))) for r in batch])
#             y = model(patches.to(device))
#             feature_list.append(y)

#     feature_list = torch.concat(feature_list,0)
#     label = 0

#     torch.save({
#         "features":feature_list,
#         "rects":rects,
#         "level":level,
#         "level_base":level_base,
#         "patch_size":p_size,
#         "level":level,
#         "label":label
#     },os.path.join(save_path,f"{f[:-4]}_{len(feature_list)}_{label}.pth"))
#     # break

In [ ]:
# 准备数据 BRACS
img_level = 20
p_size = 256
batch_size=64


device = torch.device("cuda:0")
patch_size=16
model = vit_small(patch_size=patch_size)
state_dict = torch.load(f"./model/base/DINO/{img_level}_checkpoint.pth")["teacher"]
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict,strict=False)
model.eval()
model = model.to(device)





# Normal:0, Benign:1, Insitu:2, Invasive:3 
wsi_path = "D:/TJL/testing/Invasive/"
save_path = f"../../BRACS/pkl_data/testing/"
for f in tqdm([f for f in os.listdir(wsi_path) if f.endswith("svs")]):
    wsi = WSI(os.path.join(wsi_path,f))

    level = 1
    level_base = 2
    
    _,rects,_ = wsi.get_tissue_patches(level,level_base,p_size,0.5)

    feature_list = []
    for i in range(0,len(rects),batch_size):
        with torch.no_grad():
            batch = rects[i:i+batch_size]
            patches  = torch.stack([normalize(wsi.read_region(r,level,(p_size,p_size))) for r in batch])
            y = model(patches.to(device))
            feature_list.append(y)

    feature_list = torch.concat(feature_list,0)
    label = 3

    torch.save({
        "features":feature_list,
        "rects":rects,
        "level":level,
        "level_base":level_base,
        "patch_size":p_size,
        "level":level,
        "label":label
    },os.path.join(save_path,f"{f[:-4]}_{len(feature_list)}_{label}.pth"))
    # break


In [ ]:
# wsi_path = "D:/TJL/testing/Normal/"
# levels = []
# for f in tqdm([f for f in os.listdir(wsi_path) if f.endswith("svs")]):
#     wsi = WSI(os.path.join(wsi_path,f))
#     levels.append(wsi.level_count)
# np.unique(levels)
f

In [ ]:

# nums = []
# labels = []
# for f in os.listdir("../../CAMELYON16/pkl_data/training/20xbags/")+os.listdir("../../CAMELYON16/pkl_data/testing/20xbags/"):
#     nums.append(int(f[:-4].split("_")[-2]))
#     labels.append(int(f[:-4].split("_")[-1]))
# nums = np.array(nums)
# labels = np.array(labels)
# plt.rcParams['font.sans-serif'] = ['SimHei']
# plt.figure()
# sns.histplot(nums[labels==0],label="正常")
# sns.histplot(nums[labels==1],label="肿瘤")
# plt.xlabel("patch数量")
# plt.ylabel("WSI数量")
# plt.title("level_1")
# plt.legend()
# print(f"{int(np.mean(nums))}\t{np.std(nums):.3f}")

# nums = []
# labels = []
# for f in os.listdir("../../CAMELYON16/pkl_data/training/10xbags/")+os.listdir("../../CAMELYON16/pkl_data/testing/10xbags/"):
#     nums.append(int(f[:-4].split("_")[-2]))
#     labels.append(int(f[:-4].split("_")[-1]))
# nums = np.array(nums)
# labels = np.array(labels)
# plt.rcParams['font.sans-serif'] = ['SimHei']
# plt.figure()
# sns.histplot(nums[labels==0],label="正常")
# sns.histplot(nums[labels==1],label="肿瘤")
# plt.xlabel("patch数量")
# plt.ylabel("WSI数量")
# plt.title("level_2")
# plt.legend()
# print(f"{int(np.mean(nums))}\t{np.std(nums):.3f}")

# nums = []
# labels = []
# for f in os.listdir("../../CAMELYON16/pkl_data/training/5xbags/")+os.listdir("../../CAMELYON16/pkl_data/testing/5xbags/"):
#     nums.append(int(f[:-4].split("_")[-2]))
#     labels.append(int(f[:-4].split("_")[-1]))
# nums = np.array(nums)
# labels = np.array(labels)
# plt.rcParams['font.sans-serif'] = ['SimHei']
# plt.figure()
# sns.histplot(nums[labels==0],label="正常")
# sns.histplot(nums[labels==1],label="肿瘤")
# plt.xlabel("patch数量")
# plt.ylabel("WSI数量")
# plt.title("level_3")
# plt.legend()
# print(f"{int(np.mean(nums))}\t{np.std(nums):.3f}")


